In [1]:
# ============================================================
#  09 Population / Residential Intensity / Activity Proxy
#  数据源: WorldPop 100m + 06 Buildings + 08 POI Demand
# ============================================================

from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
import rasterio
from rasterio.mask import mask as rio_mask
import matplotlib.pyplot as plt
from shapely.geometry import box, mapping
from shapely.prepared import prep as shapely_prep
import requests
import warnings
warnings.filterwarnings("ignore")

RAW = Path("data_raw")
OUT = Path("data_out")
OUT.mkdir(exist_ok=True)

BOUNDARY = Path("../04 Transport/data_raw/shenzhen_boundary.geojson")
BUILDINGS_06 = Path("../06 Buildings/data_out/sz_buildings.gpkg")
DEMAND_08 = Path("../08 POI Demand/data_out/sz_demand_grid.gpkg")

# 优先用 00 已有的广东省 WorldPop (不需要下载全国 tif)
WORLDPOP_LOCAL = Path("../00/13-人口密度-worldpop/13-人口密度-worldpop/广东省_chn_ppp_2020_UNadj.tif")
WORLDPOP_TIF = WORLDPOP_LOCAL if WORLDPOP_LOCAL.exists() else RAW / "chn_ppp_2020_constrained_100m.tif"
CLIPPED_TIF = OUT / "sz_worldpop_clipped.tif"

# 00 小区点 (补充住宅密度)
XIAOQU_PT = Path("../00/11-公服设施/11-公服设施/SHP/深圳市-小区点.shp")

shenzhen = gpd.read_file(BOUNDARY).to_crs(4326)
shenzhen_geom = shenzhen.union_all() if hasattr(shenzhen, "union_all") else shenzhen.unary_union
bbox = shenzhen_geom.bounds

print(f"06 buildings: {BUILDINGS_06.exists()}")
print(f"08 demand grid: {DEMAND_08.exists()}")
print(f"WorldPop tif: {WORLDPOP_TIF} -> exists: {WORLDPOP_TIF.exists()}")
print(f"小区点: {XIAOQU_PT.exists()}")

06 buildings: True
08 demand grid: True
WorldPop tif: ../00/13-人口密度-worldpop/13-人口密度-worldpop/广东省_chn_ppp_2020_UNadj.tif -> exists: True
小区点: True


/Users/shirly/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# ============================================================
#  1. WorldPop 人口栅格
#  优先用 00/ 已有的广东省裁剪版, 无需下载
#  如果 00 里没有, 才下载全国版
# ============================================================

if WORLDPOP_TIF.exists():
    size_mb = WORLDPOP_TIF.stat().st_size / 1e6
    print(f"Using local WorldPop: {WORLDPOP_TIF} ({size_mb:.0f} MB)")
else:
    WORLDPOP_URL = "https://data.worldpop.org/GIS/Population/Global_2000_2020_Constrained/2020/BSGM/CHN/chn_ppp_2020_constrained.tif"
    print(f"Local not found. Downloading from {WORLDPOP_URL} ...")
    r = requests.get(WORLDPOP_URL, stream=True, timeout=30)
    r.raise_for_status()
    total = int(r.headers.get("content-length", 0))
    downloaded = 0
    dl_path = RAW / "chn_ppp_2020_constrained_100m.tif"
    with open(dl_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            f.write(chunk)
            downloaded += len(chunk)
            if total > 0:
                print(f"\r  {downloaded / 1e6:.0f} / {total / 1e6:.0f} MB", end="", flush=True)
    WORLDPOP_TIF = dl_path
    print(f"\n  Saved -> {WORLDPOP_TIF}")

Using local WorldPop: ../00/13-人口密度-worldpop/13-人口密度-worldpop/广东省_chn_ppp_2020_UNadj.tif (1 MB)


In [3]:
# ============================================================
#  2. 裁剪栅格到深圳 + 转为分析网格
#  WorldPop 100m → 裁剪 → 重采样到 0.005° (~550m) 网格
# ============================================================
from rasterio.warp import calculate_default_transform, reproject, Resampling

GRID_SIZE = 0.005  # 与 06/08 一致

# ── 裁剪 WorldPop 到深圳 ──
if not CLIPPED_TIF.exists():
    print("Clipping WorldPop to Shenzhen ...")
    with rasterio.open(WORLDPOP_TIF) as src:
        geom = [mapping(shenzhen_geom)]
        out_image, out_transform = rio_mask(src, geom, crop=True, nodata=-99999)
        out_meta = src.meta.copy()
        out_meta.update({
            "driver": "GTiff",
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform,
            "nodata": -99999,
        })
        with rasterio.open(CLIPPED_TIF, "w", **out_meta) as dst:
            dst.write(out_image)
    print(f"  Saved -> {CLIPPED_TIF}")
else:
    print(f"Clipped raster exists: {CLIPPED_TIF}")

# ── 读取裁剪后栅格 ──
with rasterio.open(CLIPPED_TIF) as src:
    pop_data = src.read(1)
    pop_transform = src.transform
    pop_crs = src.crs
    print(f"  Shape: {pop_data.shape}, CRS: {pop_crs}")
    print(f"  Valid cells: {(pop_data > 0).sum():,}")
    print(f"  Total population: {pop_data[pop_data > 0].sum():,.0f}")

# ── 转为分析网格 (0.005°) ──
minx, miny, maxx, maxy = shenzhen_geom.bounds
sz_prep = shapely_prep(shenzhen_geom)

grid_cells = []
gid = 0
x = minx
while x < maxx:
    y = miny
    while y < maxy:
        cell = box(x, y, x + GRID_SIZE, y + GRID_SIZE)
        if sz_prep.intersects(cell):
            grid_cells.append({"grid_id": gid, "geometry": cell,
                               "cx": x + GRID_SIZE / 2, "cy": y + GRID_SIZE / 2})
            gid += 1
        y += GRID_SIZE
    x += GRID_SIZE

pop_grid = gpd.GeoDataFrame(grid_cells, crs=4326)

# 对每个网格, 从栅格中采样人口 (用网格中心点)
from rasterio.transform import rowcol

with rasterio.open(CLIPPED_TIF) as src:
    pop_values = []
    for _, row in pop_grid.iterrows():
        try:
            r, c = rowcol(src.transform, row["cx"], row["cy"])
            if 0 <= r < pop_data.shape[0] and 0 <= c < pop_data.shape[1]:
                val = pop_data[r, c]
                pop_values.append(max(val, 0))
            else:
                pop_values.append(0)
        except Exception:
            pop_values.append(0)

pop_grid["pop_count"] = pop_values
grid_area_km2 = (GRID_SIZE * 111.32) ** 2
pop_grid["pop_density"] = pop_grid["pop_count"] / grid_area_km2

print(f"\nGrid: {len(pop_grid):,} cells")
print(f"Cells with population > 0: {(pop_grid['pop_count'] > 0).sum():,}")
print(f"Total grid population: {pop_grid['pop_count'].sum():,.0f}")
print(f"Max density: {pop_grid['pop_density'].max():,.0f} /km²")

Clipped raster exists: data_out/sz_worldpop_clipped.tif
  Shape: (551, 1052), CRS: EPSG:4326
  Valid cells: 120,461
  Total population: 15,240,720

Grid: 7,392 cells
Cells with population > 0: 3,328
Total grid population: 423,061
Max density: 1,138 /km²


In [4]:
# ============================================================
#  3. 住宅强度 (from 06 Buildings)
#  读取已有建筑数据, 按网格汇总住宅建筑体积和栋数
# ============================================================

if BUILDINGS_06.exists():
    print("Loading buildings ...")
    bldg = gpd.read_file(BUILDINGS_06)

    # 只取住宅
    res = bldg[bldg["usage_cat"] == "residential"].copy()
    print(f"  Residential buildings: {len(res):,}")

    # 计算体积
    res["volume_m3"] = res["footprint_m2"] * res["height_m"]

    # 质心 → 空间连接到网格
    res_pts = res.copy()
    res_pts["geometry"] = res_pts.geometry.centroid
    joined = gpd.sjoin(res_pts, pop_grid[["grid_id", "geometry"]], how="inner", predicate="within")

    res_stats = joined.groupby("grid_id").agg(
        residential_count=("height_m", "count"),
        residential_volume=("volume_m3", "sum"),
        residential_avg_height=("height_m", "mean"),
    ).reset_index()

    pop_grid = pop_grid.merge(res_stats, on="grid_id", how="left")
    pop_grid[["residential_count", "residential_volume", "residential_avg_height"]] = \
        pop_grid[["residential_count", "residential_volume", "residential_avg_height"]].fillna(0)

    del bldg, res, res_pts, joined
    print(f"  Grids with residential buildings: {(pop_grid['residential_count'] > 0).sum():,}")
else:
    print("06 Buildings not found, skipping residential intensity")
    pop_grid["residential_count"] = 0
    pop_grid["residential_volume"] = 0
    pop_grid["residential_avg_height"] = 0

# ── 补充: 00 小区点 → 小区密度 ──
if XIAOQU_PT.exists():
    print("\nLoading 小区点 ...")
    xq_pts = gpd.read_file(XIAOQU_PT).to_crs(4326)
    xq_pts = gpd.clip(xq_pts, shenzhen_geom)
    print(f"  小区点: {len(xq_pts):,}")

    xq_joined = gpd.sjoin(xq_pts, pop_grid[["grid_id", "geometry"]], how="inner", predicate="within")
    xq_stats = xq_joined.groupby("grid_id").size().reset_index(name="xiaoqu_count")
    pop_grid = pop_grid.merge(xq_stats, on="grid_id", how="left")
    pop_grid["xiaoqu_count"] = pop_grid["xiaoqu_count"].fillna(0).astype(int)
    del xq_pts, xq_joined
    print(f"  Grids with 小区: {(pop_grid['xiaoqu_count'] > 0).sum():,}")
else:
    pop_grid["xiaoqu_count"] = 0
    print("小区点 not found, skipping")

Loading buildings ...
  Residential buildings: 402,059
  Grids with residential buildings: 4,618

Loading 小区点 ...
  小区点: 12,125
  Grids with 小区: 2,790


In [5]:
# ============================================================
#  3b. 百度热力图 — 周末/节假日活动热点 proxy
#  用百度地图 热力图 API 抓取人群密度快照
#  API: https://lbsyun.baidu.com/faq/api?title=webapi-heat
#  注: 百度热力图 API 返回的是实时快照, 建议周末下午抓一次
# ============================================================

BAIDU_AK = "WBjHYGJcg9il6cUXt5YzFwsFqGc63Er7"
HEATMAP_CACHE = RAW / "baidu_heatmap_cache.json"

import json, time

def fetch_baidu_heatmap(city="深圳", ak=BAIDU_AK):
    """抓取百度地图城市热力图数据 (实时人群密度)"""
    url = "https://huiyan.baidu.com/migration/cityrank.jsonp"
    params = {
        "dt": "city",
        "id": "440300",
        "type": "move",
        "date": time.strftime("%Y%m%d"),
        "callback": "",
    }
    try:
        r = requests.get(url, params=params, timeout=15)
        text = r.text.strip()
        if text.startswith("("):
            text = text[1:-1]
        return json.loads(text)
    except Exception as e:
        print(f"  Baidu heatmap error: {e}")
        return None

# 百度热力图的公开 API 有限制, 替代方案:
# 用百度 Place API 搜索景区/商圈/公园的评论数/评分作为活动 proxy
def fetch_baidu_poi_activity(query, region="深圳", ak=BAIDU_AK, tag="旅游景点"):
    """用百度地点搜索 API 获取 POI 及评论数作为活动 proxy"""
    url = "https://api.map.baidu.com/place/v2/search"
    all_pois = []
    page = 0
    while True:
        params = {
            "query": query,
            "region": region,
            "output": "json",
            "ak": ak,
            "page_size": 20,
            "page_num": page,
            "scope": 2,
        }
        try:
            r = requests.get(url, params=params, timeout=15)
            data = r.json()
            if data.get("status") != 0:
                break
            results = data.get("results", [])
            if not results:
                break
            for p in results:
                loc = p.get("location", {})
                detail = p.get("detail_info", {})
                all_pois.append({
                    "name": p.get("name", ""),
                    "lon": loc.get("lng", 0),
                    "lat": loc.get("lat", 0),
                    "overall_rating": float(detail.get("overall_rating", 0)),
                    "comment_num": int(detail.get("comment_num", 0)),
                    "tag": tag,
                })
            page += 1
            time.sleep(0.3)
            if page >= 20:
                break
        except Exception as e:
            print(f"  Error: {e}")
            break
    return all_pois

# 抓取深圳热门场景的活动数据
if HEATMAP_CACHE.exists():
    print(f"Loading cache: {HEATMAP_CACHE}")
    with open(HEATMAP_CACHE) as f:
        activity_pois = json.load(f)
else:
    print("Fetching Baidu POI activity data ...")
    activity_pois = []
    for query, tag in [("公园", "park"), ("景区", "scenic"), ("商圈", "commercial"),
                        ("购物中心", "mall"), ("步行街", "street")]:
        pois = fetch_baidu_poi_activity(query, tag=tag)
        activity_pois.extend(pois)
        print(f"  {tag}: {len(pois)} POIs")
        time.sleep(0.5)

    with open(HEATMAP_CACHE, "w") as f:
        json.dump(activity_pois, f, ensure_ascii=False)
    print(f"Total: {len(activity_pois)} activity POIs, cached.")

# 转 GeoDataFrame
if activity_pois:
    act_df = pd.DataFrame(activity_pois)
    act_gdf = gpd.GeoDataFrame(
        act_df, geometry=gpd.points_from_xy(act_df["lon"], act_df["lat"]), crs=4326
    )
    act_gdf = gpd.clip(act_gdf, shenzhen_geom)
    # 注: 百度坐标 BD09 → WGS84 有偏移, 此处近似使用
    print(f"  Activity POIs in Shenzhen: {len(act_gdf)}")

    # 空间连接到 pop_grid
    act_joined = gpd.sjoin(act_gdf, pop_grid[["grid_id", "geometry"]], how="inner", predicate="within")
    act_stats = act_joined.groupby("grid_id").agg(
        weekend_hotspot_count=("comment_num", "count"),
        weekend_hotspot_score=("comment_num", "sum"),
    ).reset_index()
    pop_grid = pop_grid.merge(act_stats, on="grid_id", how="left")
    pop_grid[["weekend_hotspot_count", "weekend_hotspot_score"]] = \
        pop_grid[["weekend_hotspot_count", "weekend_hotspot_score"]].fillna(0)
    print(f"  Grids with hotspots: {(pop_grid['weekend_hotspot_count'] > 0).sum()}")
else:
    pop_grid["weekend_hotspot_count"] = 0
    pop_grid["weekend_hotspot_score"] = 0
    print("  No activity data available")

Loading cache: data_raw/baidu_heatmap_cache.json
  Activity POIs in Shenzhen: 466
  Grids with hotspots: 354


In [6]:
# ============================================================
#  4. 合并 08 活动强度 + 合成 intensity_index + 保存
# ============================================================

# ── 合并 08 demand_pressure ──
if DEMAND_08.exists():
    print("Loading 08 demand grid ...")
    demand = gpd.read_file(DEMAND_08)

    # 取 demand_pressure 列, 用网格中心点匹配
    demand["cx"] = demand.geometry.centroid.x.round(5)
    demand["cy"] = demand.geometry.centroid.y.round(5)
    pop_grid["cx_r"] = pop_grid["cx"].round(5)
    pop_grid["cy_r"] = pop_grid["cy"].round(5)

    demand_map = demand.set_index(["cx", "cy"])
    dp_col = "demand_pressure" if "demand_pressure" in demand.columns else "poi_total"

    # 空间连接 (用质心最近匹配)
    demand_pts = demand[["geometry", dp_col]].copy()
    demand_pts["geometry"] = demand_pts.geometry.centroid
    pop_pts = pop_grid[["grid_id", "geometry"]].copy()
    pop_pts["geometry"] = pop_grid.geometry.centroid

    dj = gpd.sjoin_nearest(pop_pts, demand_pts, how="left", max_distance=0.01)
    pop_grid["demand_pressure"] = dj[dp_col].fillna(0).values

    # 就业中心 proxy = office POI 密度
    if "office_count" in demand.columns:
        dj2 = gpd.sjoin_nearest(
            pop_grid[["grid_id", "geometry"]].copy().set_geometry(pop_grid.geometry.centroid),
            demand[["geometry", "office_count"]].copy().set_geometry(demand.geometry.centroid),
            how="left", max_distance=0.01,
        )
        pop_grid["employment_proxy"] = dj2["office_count"].fillna(0).values
        print(f"  Merged employment_proxy")
    else:
        pop_grid["employment_proxy"] = 0

    # 商圈活跃 proxy = food + retail POI 密度
    food_col = "food_count" if "food_count" in demand.columns else None
    retail_col = "retail_count" if "retail_count" in demand.columns else None
    if food_col or retail_col:
        cols_to_sum = [c for c in [food_col, retail_col] if c]
        dj3 = gpd.sjoin_nearest(
            pop_grid[["grid_id", "geometry"]].copy().set_geometry(pop_grid.geometry.centroid),
            demand[["geometry"] + cols_to_sum].copy().set_geometry(demand.geometry.centroid),
            how="left", max_distance=0.01,
        )
        pop_grid["commercial_activity"] = sum(dj3[c].fillna(0).values for c in cols_to_sum)
        print(f"  Merged commercial_activity")
    else:
        pop_grid["commercial_activity"] = 0

    del demand, demand_pts, pop_pts, dj
    print(f"  Merged demand_pressure")
else:
    print("08 demand grid not found, skipping")
    pop_grid["demand_pressure"] = 0

# ── 合成 intensity_index ──
def norm(s):
    mn, mx = s.min(), s.max()
    return (s - mn) / (mx - mn) if mx > mn else s * 0

pop_grid["pop_norm"] = norm(pop_grid["pop_count"])
pop_grid["res_norm"] = norm(pop_grid["residential_volume"])
pop_grid["demand_norm"] = norm(pop_grid["demand_pressure"])

W_POP = 0.40
W_RES = 0.30
W_DEMAND = 0.30

pop_grid["intensity_index"] = (
    W_POP * pop_grid["pop_norm"]
  + W_RES * pop_grid["res_norm"]
  + W_DEMAND * pop_grid["demand_norm"]
)

# ── 保存 ──
out_cols = ["grid_id", "pop_count", "pop_density",
            "residential_count", "residential_volume", "residential_avg_height",
            "demand_pressure", "employment_proxy", "commercial_activity",
            "intensity_index", "geometry"]
result = pop_grid[[c for c in out_cols if c in pop_grid.columns]]
result.to_file(OUT / "sz_population_grid.gpkg", driver="GPKG")
print(f"\nSaved -> data_out/sz_population_grid.gpkg  ({len(result):,} cells)")

# ── 统计摘要 ──
g = result[result["pop_count"] > 0]
print(f"\n=== Summary (cells with pop > 0: {len(g):,}) ===")
print(f"Population:   total={g['pop_count'].sum():,.0f}  mean={g['pop_count'].mean():.0f}  max={g['pop_count'].max():.0f}")
print(f"Pop density:  mean={g['pop_density'].mean():,.0f} /km²  max={g['pop_density'].max():,.0f} /km²")
print(f"Res volume:   mean={g['residential_volume'].mean():,.0f} m³  max={g['residential_volume'].max():,.0f} m³")
print(f"Intensity:    mean={g['intensity_index'].mean():.3f}  max={g['intensity_index'].max():.3f}")

print("\n=== Top 10 Highest Intensity Grids ===")
top10 = result.nlargest(10, "intensity_index")[["grid_id", "pop_count", "residential_count", "demand_pressure", "intensity_index"]]
print(top10.to_string(index=False))

Loading 08 demand grid ...
  Merged employment_proxy
  Merged commercial_activity
  Merged demand_pressure

Saved -> data_out/sz_population_grid.gpkg  (7,392 cells)

=== Summary (cells with pop > 0: 3,328) ===
Population:   total=423,061  mean=127  max=352
Pop density:  mean=410 /km²  max=1,138 /km²
Res volume:   mean=8,363,287 m³  max=330,839,307 m³
Intensity:    mean=0.173  max=0.710

=== Top 10 Highest Intensity Grids ===
 grid_id  pop_count  residential_count  demand_pressure  intensity_index
    3892 341.750732               48.0           305.00         0.709812
    3769 265.348022               11.0            45.25         0.645676
    2198 303.730499               11.0           149.80         0.591067
    3891 322.337860               24.0           173.15         0.567602
    3890 324.422119               73.0           147.95         0.564081
    3589 321.096497                6.0           198.20         0.563893
    3930 327.798492              223.0           115.10     